# 📊 Results Preview: Distributed Simulation Output

This notebook reads the massive simulation results from **Cloud Storage (GCS)** and generates a technical visualization of the photon point cloud around the black hole.

In [ ]:
# 1. Initialize stable environment for remote rendering
%matplotlib inline
import matplotlib
matplotlib.use('Agg') # Stable backend for Dataproc containers
import matplotlib.pyplot as plt
from IPython.display import Image, display
import numpy as np

print("Visualization libraries initialized.")

In [ ]:
# 2. Connect to the Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ResultsPreview-Phase2") \
    .getOrCreate()

print("Spark Session active. Ready to read data.")

In [ ]:
# 3. Load Simulation Data from GCS
BUCKET_NAME = "black-hole-visualizer-project-bh-vis-dataproc-config"
DATA_PATH = f"gs://{BUCKET_NAME}/sim_results/phase2_test"

print(f"Reading data from: {DATA_PATH}")

try:
    df = spark.read.parquet(DATA_PATH)
    total_points = df.count()
    print(f"✅ Successfully loaded {total_points:,} coordinates.")
    
    # Sample a manageable subset for local plotting (e.g., 50,000 points)
    sample_df = df.sample(False, fraction=min(1.0, 50000/total_points)).toPandas()
    print(f"Sampled {len(sample_df)} points for local visualization.")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("HINT: Ensure the simulation job has completed successfully first.")

In [ ]:
# 4. Generate Point Cloud Visualization (2D Projection)
if 'sample_df' in locals():
    # Convert Boyer-Lindquist to Quasi-Cartesian
    r = sample_df['r']
    phi = sample_df['phi']
    x = r * np.cos(phi)
    y = r * np.sin(phi)
    
    # Visualization
    plt.figure(figsize=(12, 12), facecolor='black')
    ax = plt.gca()
    ax.set_facecolor('black')
    
    # Scatter plot with high transparency to show density
    plt.scatter(x, y, s=0.1, color='gold', alpha=0.1, label='Photon Nimbus')
    
    # Representing the Black Hole
    RS = 2.0
    circle = plt.Circle((0, 0), RS, color='white', fill=True, label='Event Horizon')
    ax.add_patch(circle)
    
    plt.xlim(-30, 30)
    plt.ylim(-30, 30)
    plt.axis('off')
    plt.title("Massive Simulation Preview - Photon Nimbus", color='white', fontsize=18)
    
    # Save and display
    plt.savefig('result_preview.png', dpi=150, bbox_inches='tight', facecolor='black')
    plt.close() # Clean up memory
    
    display(Image('result_preview.png'))
    print("✅ Visualization generated successfully.")
